# Receipt KIE Baseline: LayoutLMv3 with Word-Level CRF

This notebook trains and evaluates a reproducible baseline for key information extraction (KIE) on the CORD receipt dataset. The model combines pretrained LayoutLMv3 representations with a linear word classifier and a word-level conditional random field (CRF).

The experimental pipeline includes:

- Oracle CORD text and bounding boxes with `apply_ocr=False`.
- Word-level representation pooling and sequence decoding, excluding special tokens and continuation subtokens from supervision.
- Separate learning rates for the pretrained backbone and task-specific head.
- One epoch of backbone freezing followed by end-to-end fine-tuning.
- Capped class balancing and document-level rare-class sampling.
- Checkpoint selection based exclusively on development-set Entity Macro F1.
- Independent training with seeds `13`, `42`, `2026`, `7`, and `123`, reported as mean ± sample standard deviation.
- Reproducibility artifacts containing model checkpoints, processor files, label mappings, metrics, configuration metadata, environment versions, and SHA-256 checksums.

The primary metric is macro F1 over 18 semantic entity classes, excluding the background label `O`. Test evaluation is performed only after the best checkpoint for each seed has been selected on the development split.

**Execution:** Enable a GPU in Kaggle, attach the CORD-1000 dataset, select the `paper` profile, run all cells, and download `receipt_kie_baseline_artifacts.zip` from the notebook output.


## 1. Runtime Setup and Environment Validation

This section installs only the dependencies that are unavailable in the active runtime, imports the required libraries, verifies CUDA availability, locates the CORD dataset, and records the software environment used by the experiment. The existing PyTorch installation is retained to preserve compatibility with the Kaggle CUDA runtime.


In [3]:
import importlib.util
import subprocess
import sys

REQUIRED = {
    "torchcrf": "pytorch-crf>=0.7.2,<1",
    "transformers": "transformers>=4.30,<6",
    "sklearn": "scikit-learn>=1.2,<2",
}

for import_name, package in REQUIRED.items():
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("Dependencies are ready. Restart the kernel only if Kaggle requests it.")

Dependencies are ready. Restart the kernel only if Kaggle requests it.


In [4]:
import gc
import hashlib
import json
import math
import os
import platform
import random
import shutil
import time
from collections import Counter
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_recall_fscore_support
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchcrf import CRF
from tqdm.auto import tqdm
from transformers import LayoutLMv3Model, LayoutLMv3Processor, get_linear_schedule_with_warmup

assert torch.cuda.is_available(), "Hãy bật GPU trong Kaggle trước khi train."

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("Transformers:", transformers.__version__)
print("scikit-learn:", sklearn.__version__)

Python: 3.12.13
PyTorch: 2.10.0+cu128
CUDA: 12.8
GPU: Tesla T4
Transformers: 5.0.0
scikit-learn: 1.6.1


## 2. Experimental Configuration

This section defines the complete baseline configuration, including optimization parameters, regularization, class-balancing controls, early stopping, and artifact paths.

Three execution profiles are available:

- `smoke`: one short run for pipeline validation.
- `standard`: one complete training run with seed `42`.
- `paper`: five independent runs with seeds `13`, `42`, `2026`, `7`, and `123`.

For every seed, model initialization starts from the same pretrained LayoutLMv3 checkpoint. The best checkpoint is selected using development-set Entity Macro F1, and the test split is not used during training or model selection.


In [5]:
@dataclass(frozen=True)
class Config:
    profile: str = "paper"
    model_id: str = "microsoft/layoutlmv3-base"
    model_revision: str = "main"
    max_length: int = 512
    batch_size: int = 2
    grad_accum_steps: int = 2
    num_workers: int = 2

    backbone_lr: float = 1e-5
    head_lr: float = 1e-4
    weight_decay: float = 0.01
    warmup_ratio: float = 0.10
    max_epochs: int = 14
    patience: int = 4
    min_delta: float = 1e-4
    freeze_backbone_epochs: int = 1
    lm_dropout: float = 0.30
    crf_weight: float = 0.65
    focal_gamma: float = 2.0
    use_rare_document_sampler: bool = True
    rare_sampler_power: float = 0.50
    rare_sampler_max_weight: float = 2.50
    class_balance_beta: float = 0.999
    class_weight_cap: float = 3.0
    use_amp: bool = True
    keep_all_checkpoints: bool = True
    artifact_dir: str = "/kaggle/working/receipt_kie_baseline_artifacts"


CFG = Config()
PROFILE = {
    "smoke": {"seeds": [42], "max_epochs": 1, "patience": 1},
    "standard": {"seeds": [42], "max_epochs": CFG.max_epochs, "patience": CFG.patience},
    "paper": {"seeds": [13, 42, 2026, 7, 123], "max_epochs": CFG.max_epochs, "patience": CFG.patience},
}[CFG.profile]

SEEDS = PROFILE["seeds"]
MAX_EPOCHS = PROFILE["max_epochs"]
PATIENCE = PROFILE["patience"]
MODEL_NAME = "layoutlmv3_word_crf"
DEVICE = torch.device("cuda")
ARTIFACT_DIR = Path(CFG.artifact_dir)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print("Profile:", CFG.profile, "| seeds:", SEEDS, "| max epochs:", MAX_EPOCHS)


Profile: paper | seeds: [13, 42, 2026, 7, 123] | max epochs: 14


In [6]:
def set_seed(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def find_cord_root() -> Path:
    candidates = [Path("CORD1000/CORD/CORD")]
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        candidates.extend(p.parent.parent for p in kaggle_input.rglob("train/json"))
    for root in candidates:
        if all((root / split / "json").is_dir() and (root / split / "image").is_dir()
               for split in ("train", "dev", "test")):
            return root.resolve()
    raise FileNotFoundError("Không tìm thấy CORD root có train/dev/test/{image,json}. Hãy attach dataset vào Kaggle.")


set_seed(SEEDS[0])
CORD_ROOT = find_cord_root()
print("CORD root:", CORD_ROOT)


CORD root: /kaggle/input/datasets/lonelvino/cord-1000/CORD/CORD


## 3. Label Schema and Dataset Audit

This section maps CORD annotations to 18 receipt entity classes and the background class `O`. It also audits the train, development, and test splits by checking image–annotation pairing, document counts, word counts, and label frequencies.

The primary evaluation metric is Entity Macro F1 computed over the 18 entity labels while excluding `O`. Entity Micro F1, weighted F1, precision, recall, and per-class support are retained as complementary diagnostics.


In [7]:
CORD_CATEGORY_TO_LABEL = {
    "menu.nm": "S-MENU_NM", "menu.sub_nm": "S-MENU_NM",
    "menu.cnt": "S-MENU_CNT", "menu.sub_cnt": "S-MENU_CNT",
    "menu.num": "S-MENU_NUM", "menu.unitprice": "S-MENU_UNITPRICE",
    "menu.price": "S-MENU_PRICE", "menu.sub_price": "S-MENU_PRICE",
    "menu.discountprice": "S-MENU_DISCOUNT_PRICE",
    "sub_total.subtotal_price": "S-SUBTOTAL",
    "sub_total.discount_price": "S-DISCOUNT",
    "sub_total.tax_price": "S-TAX", "sub_total.service_price": "S-SERVICE",
    "total.total_price": "S-TOTAL", "total.cashprice": "S-CASH",
    "total.changeprice": "S-CHANGE", "total.creditcardprice": "S-CARD_PAYMENT",
    "total.emoneyprice": "S-EMONEY_PAYMENT", "total.menuqty_cnt": "S-MENUQTY_CNT",
    "total.menutype_cnt": "S-MENUTYPE_CNT",
    "sub_total.etc": "S-OTHER", "total.total_etc": "S-OTHER",
    "menu.etc": "S-OTHER", "menu.sub_etc": "S-OTHER",
    "menu.itemsubtotal": "O", "menu.vatyn": "O",
    "sub_total.othersvc_price": "O", "void_menu.nm": "O",
    "void_menu.price": "O", "menu.sub_unitprice": "O",
}

LABELS = [
    "O", "S-MENU_NM", "S-MENU_CNT", "S-MENU_NUM", "S-MENU_UNITPRICE",
    "S-MENU_PRICE", "S-MENU_DISCOUNT_PRICE", "S-SUBTOTAL", "S-DISCOUNT",
    "S-TAX", "S-SERVICE", "S-TOTAL", "S-CASH", "S-CHANGE",
    "S-CARD_PAYMENT", "S-EMONEY_PAYMENT", "S-MENUQTY_CNT",
    "S-MENUTYPE_CNT", "S-OTHER",
]
ID2LABEL = dict(enumerate(LABELS))
LABEL2ID = {label: idx for idx, label in ID2LABEL.items()}
ENTITY_IDS = list(range(1, len(LABELS)))
NUM_LABELS = len(LABELS)


def dataset_audit(root: Path) -> pd.DataFrame:
    rows = []
    for split in ("train", "dev", "test"):
        json_files = sorted((root / split / "json").glob("*.json"))
        image_files = sorted((root / split / "image").glob("*.png"))
        json_stems, image_stems = {p.stem for p in json_files}, {p.stem for p in image_files}
        counts = Counter()
        for path in json_files:
            data = json.loads(path.read_text(encoding="utf-8"))
            for line in data.get("valid_line", []):
                mapped = CORD_CATEGORY_TO_LABEL.get(line.get("category", ""), "O")
                counts[mapped] += len(line.get("words", []))
        rows.append({
            "split": split, "json": len(json_files), "images": len(image_files),
            "missing_images": len(json_stems - image_stems),
            "missing_json": len(image_stems - json_stems),
            "words": sum(counts.values()), **counts,
        })
    return pd.DataFrame(rows).fillna(0)


audit_df = dataset_audit(CORD_ROOT)
display(audit_df)
assert (audit_df[["missing_images", "missing_json"]].to_numpy() == 0).all(), "Dataset thiếu cặp image/json."


,split,json,images,missing_images,missing_json,words,S-MENU_CNT,S-MENU_NM,S-MENU_PRICE,S-SUBTOTAL,...,S-CHANGE,S-MENUTYPE_CNT,S-MENUQTY_CNT,S-DISCOUNT,S-MENU_UNITPRICE,S-CARD_PAYMENT,S-MENU_NUM,S-MENU_DISCOUNT_PRICE,S-EMONEY_PAYMENT,O
0,train,800,800,0,0,19371,2126,5995,2236,1187,...,1044,105,513,164,629,326,94,355,115,31
1,dev,100,100,0,0,2186,246,686,241,151,...,135,8,50,11,52,34,4,18,10,3
2,test,100,100,0,0,2356,246,740,268,145,...,120,17,67,16,69,51,11,30,4,6


## 4. CORD Record Construction and Word-Level Encoding

This section converts CORD JSON annotations into document records containing text, labels, images, and word bounding boxes. Quadrilateral annotations are converted to rectangular boxes and normalized to LayoutLMv3's `[0, 1000]` coordinate space.

The LayoutLMv3 processor receives the reference text and bounding boxes with OCR disabled. Subtoken outputs are mapped back to their source words, and only words retained after truncation are included in the word-level targets. The data loader pads variable-length word sequences and applies deterministic document sampling with bounded emphasis on receipts containing rare entity classes.


In [8]:
def quad_to_bbox(quad, width, height):
    xs = [quad[f"x{i}"] for i in range(1, 5)]
    ys = [quad[f"y{i}"] for i in range(1, 5)]
    x0, y0, x1, y1 = int(min(xs)), int(min(ys)), int(max(xs)), int(max(ys))
    x0, y0 = max(0, min(x0, width - 1)), max(0, min(y0, height - 1))
    x1, y1 = max(x0 + 1, min(x1, width)), max(y0 + 1, min(y1, height))
    return x0, y0, x1, y1


def build_records(root: Path, split: str):
    records = []
    for json_path in tqdm(sorted((root / split / "json").glob("*.json")), desc=f"Parse {split}"):
        data = json.loads(json_path.read_text(encoding="utf-8"))
        width = int(data["meta"]["image_size"]["width"])
        height = int(data["meta"]["image_size"]["height"])
        image_path = root / split / "image" / f"{json_path.stem}.png"
        words = []
        for line in data.get("valid_line", []):
            label = CORD_CATEGORY_TO_LABEL.get(line.get("category", ""), "O")
            for word in line.get("words", []):
                text = word.get("text", "").strip()
                if text:
                    words.append({
                        "text": text,
                        "box": quad_to_bbox(word["quad"], width, height),
                        "label": label,
                    })
        words.sort(key=lambda item: (item["box"][1], item["box"][0]))
        if words and image_path.exists():
            records.append({
                "id": json_path.stem, "image_path": image_path,
                "size": (width, height), "words": words,
            })
    return records


train_records = build_records(CORD_ROOT, "train")
dev_records = build_records(CORD_ROOT, "dev")
test_records = build_records(CORD_ROOT, "test")
print("Documents:", len(train_records), len(dev_records), len(test_records))


Parse train:   0%|          | 0/800 [00:00<?, ?it/s]

Parse dev:   0%|          | 0/100 [00:00<?, ?it/s]

Parse test:   0%|          | 0/100 [00:00<?, ?it/s]

Documents: 800 100 100


In [9]:
def normalize_box(box, width, height):
    x0, y0, x1, y1 = box
    values = [1000 * x0 / width, 1000 * y0 / height, 1000 * x1 / width, 1000 * y1 / height]
    return [max(0, min(1000, int(v))) for v in values]

In [10]:
processor = LayoutLMv3Processor.from_pretrained(
    CFG.model_id, revision=CFG.model_revision, apply_ocr=False
)


class CORDWordDataset(Dataset):
    def __init__(self, records):
        self.records = records

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records[index]
        width, height = record["size"]
        image = Image.open(record["image_path"]).convert("RGB")
        texts = [item["text"] for item in record["words"]]
        boxes = [normalize_box(item["box"], width, height) for item in record["words"]]
        encoding = processor(
            image, texts, boxes=boxes, truncation=True, max_length=CFG.max_length,
            padding="max_length", return_tensors="pt",
        )
        word_ids = encoding.word_ids(batch_index=0)
        active_ids = sorted({wid for wid in word_ids if wid is not None})
        ranges = []
        for wid in active_ids:
            positions = [pos for pos, current in enumerate(word_ids) if current == wid]
            ranges.append((positions[0], positions[-1] + 1))
        active_words = [record["words"][wid] for wid in active_ids]
        labels = torch.tensor([LABEL2ID[item["label"]] for item in active_words], dtype=torch.long)
        return {
            "id": record["id"],
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "bbox": encoding["bbox"].squeeze(0),
            "pixel_values": encoding["pixel_values"].squeeze(0),
            "word_ranges": ranges, "word_labels": labels,
        }


def collate_fn(samples):
    tensor_keys = ("input_ids", "attention_mask", "bbox", "pixel_values")
    batch = {key: torch.stack([sample[key] for sample in samples]) for key in tensor_keys}
    max_words = max(len(sample["word_labels"]) for sample in samples)
    labels = torch.zeros((len(samples), max_words), dtype=torch.long)
    mask = torch.zeros((len(samples), max_words), dtype=torch.bool)
    for i, sample in enumerate(samples):
        size = len(sample["word_labels"])
        labels[i, :size] = sample["word_labels"]
        mask[i, :size] = True
    batch.update({
        "ids": [sample["id"] for sample in samples],
        "word_ranges": [sample["word_ranges"] for sample in samples],
        "word_labels": labels, "word_mask": mask,
    })
    return batch


train_ds, dev_ds, test_ds = map(CORDWordDataset, (train_records, dev_records, test_records))


def build_document_sample_weights(records):
    document_labels = [
        {LABEL2ID[word["label"]] for word in record["words"] if word["label"] != "O"}
        for record in records
    ]
    document_frequency = Counter(label for labels in document_labels for label in labels)
    weights = []
    for labels in document_labels:
        rarity = [
            (len(records) / max(document_frequency[label], 1)) ** CFG.rare_sampler_power
            for label in labels
        ]
        weights.append(min(max(rarity, default=1.0), CFG.rare_sampler_max_weight))
    return torch.tensor(weights, dtype=torch.double), document_frequency


train_sample_weights, train_document_frequency = build_document_sample_weights(train_records)
print("Rare-document sampler:", CFG.use_rare_document_sampler,
      "| weight range:", f"{train_sample_weights.min():.2f}–{train_sample_weights.max():.2f}")


def make_loaders(seed):
    loader_generator = torch.Generator().manual_seed(seed)
    common = dict(batch_size=CFG.batch_size, num_workers=CFG.num_workers,
                  pin_memory=True, collate_fn=collate_fn,
                  persistent_workers=CFG.num_workers > 0)
    if CFG.use_rare_document_sampler:
        sampler_generator = torch.Generator().manual_seed(seed)
        sampler = WeightedRandomSampler(
            train_sample_weights, num_samples=len(train_ds), replacement=True,
            generator=sampler_generator,
        )
        train_loader = DataLoader(train_ds, sampler=sampler, generator=loader_generator, **common)
    else:
        train_loader = DataLoader(train_ds, shuffle=True, generator=loader_generator, **common)
    return (
        train_loader,
        DataLoader(dev_ds, shuffle=False, **common),
        DataLoader(test_ds, shuffle=False, **common),
    )

preprocessor_config.json:   0%|          | 0.00/275 [00:00<?, ?B/s]

The image processor of type `LayoutLMv3ImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json:   0%|          | 0.00/856 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Rare-document sampler: True | weight range: 1.05–2.50


## 5. Baseline Model Architecture

The baseline consists of four components:

1. A pretrained LayoutLMv3 backbone encodes image, text, and two-dimensional layout information.
2. Subtoken representations belonging to the same word are mean-pooled into one word representation.
3. Dropout and a linear classifier produce word-level emission scores.
4. A word-level CRF models dependencies between adjacent output labels and performs structured sequence decoding.

The supervised objective combines CRF negative log-likelihood with class-weighted focal cross-entropy. This formulation preserves structured decoding while reducing the influence of dominant, easy classes.


In [11]:
class WordModelBase(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = LayoutLMv3Model.from_pretrained(CFG.model_id, revision=CFG.model_revision)
        self.hidden_size = self.backbone.config.hidden_size
        self.crf = CRF(NUM_LABELS, batch_first=True)

    @staticmethod
    def aggregate_words(token_embeddings, ranges_per_doc):
        docs = []
        for batch_index, ranges in enumerate(ranges_per_doc):
            docs.append(torch.stack([
                token_embeddings[batch_index, start:end].mean(dim=0)
                for start, end in ranges
            ]))
        return docs

    @staticmethod
    def pad_documents(doc_embeddings, max_words):
        hidden = doc_embeddings[0].shape[-1]
        padded = doc_embeddings[0].new_zeros((len(doc_embeddings), max_words, hidden))
        for i, embeddings in enumerate(doc_embeddings):
            padded[i, :len(embeddings)] = embeddings
        return padded

    def encode(self, batch):
        outputs = self.backbone(
            input_ids=batch["input_ids"].to(DEVICE),
            attention_mask=batch["attention_mask"].to(DEVICE),
            bbox=batch["bbox"].to(DEVICE),
            pixel_values=batch["pixel_values"].to(DEVICE),
        )
        text_length = batch["input_ids"].shape[1]
        return self.aggregate_words(outputs.last_hidden_state[:, :text_length], batch["word_ranges"])

    def auxiliary_loss(self):
        return next(self.parameters()).new_zeros(())

    def loss(self, emissions, labels, mask, class_weights):
        emissions_fp32 = emissions.float()
        crf_loss = -self.crf(emissions_fp32, labels, mask=mask, reduction="token_mean")
        logits, targets = emissions_fp32[mask], labels[mask]
        ce = F.cross_entropy(logits, targets, weight=class_weights, reduction="none")
        pt = torch.softmax(logits, dim=-1).gather(1, targets[:, None]).squeeze(1)
        focal_loss = (((1 - pt) ** CFG.focal_gamma) * ce).mean()
        supervised_loss = CFG.crf_weight * crf_loss + (1 - CFG.crf_weight) * focal_loss
        return supervised_loss + self.auxiliary_loss()

    def decode(self, emissions, mask):
        return self.crf.decode(emissions.float(), mask=mask)

class LayoutLMv3WordCRF(WordModelBase):
    def __init__(self):
        super().__init__()
        self.dropout = nn.Dropout(CFG.lm_dropout)
        self.classifier = nn.Linear(self.hidden_size, NUM_LABELS)

    def set_backbone_trainable(self, trainable):
        for parameter in self.backbone.parameters():
            parameter.requires_grad = trainable
        if not trainable:
            self.backbone.eval()

    def forward(self, batch):
        lm_docs = self.encode(batch)
        lm_words = self.pad_documents(lm_docs, batch["word_mask"].shape[1])
        return self.classifier(self.dropout(lm_words))

## 6. Optimization, Metrics, and Checkpoint Selection

This section computes class weights exclusively from the training split using effective-number weighting with an upper cap. The optimizer uses separate parameter groups for the LayoutLMv3 backbone and the classification/CRF head, followed by linear warmup and learning-rate decay.

The backbone is frozen during the first epoch and then unfrozen without rebuilding the optimizer or scheduler. Mixed-precision training, gradient accumulation, gradient clipping, and early stopping are applied consistently across seeds. At the end of each epoch, the model is evaluated on the development split; a checkpoint is saved only when Entity Macro F1 improves. The test loader is never accessed inside the training loop.


In [12]:
train_label_counts = Counter(
    LABEL2ID[word["label"]] for record in train_records for word in record["words"]
)
counts = torch.tensor([train_label_counts.get(i, 0) for i in range(NUM_LABELS)], dtype=torch.float32)
beta = CFG.class_balance_beta
effective_counts = (1 - torch.pow(torch.tensor(beta), counts.clamp_min(1))) / (1 - beta)
class_weights = effective_counts.reciprocal()
entity_mean = class_weights[1:].mean()
class_weights = class_weights / entity_mean
class_weights[0] = 1.0
class_weights = class_weights.clamp(max=CFG.class_weight_cap).to(DEVICE)
display(pd.DataFrame({"label": LABELS, "train_words": counts.int(), "weight": class_weights.cpu()}))


def calculate_metrics(golds, preds):
    precision, recall, macro_f1, _ = precision_recall_fscore_support(
        golds, preds, labels=ENTITY_IDS, average="macro", zero_division=0
    )
    return {
        "entity_macro_precision": float(precision),
        "entity_macro_recall": float(recall),
        "entity_macro_f1": float(macro_f1),
        "entity_micro_f1": float(f1_score(golds, preds, labels=ENTITY_IDS, average="micro", zero_division=0)),
        "weighted_f1": float(f1_score(golds, preds, labels=list(range(NUM_LABELS)), average="weighted", zero_division=0)),
    }


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    if hasattr(model, "reset_graph_diagnostics"):
        model.reset_graph_diagnostics()
    losses, golds, preds = [], [], []
    for batch in tqdm(loader, leave=False, desc="evaluate"):
        labels = batch["word_labels"].to(DEVICE)
        mask = batch["word_mask"].to(DEVICE)
        with torch.amp.autocast(device_type="cuda", enabled=CFG.use_amp):
            emissions = model(batch)
            loss = model.loss(emissions, labels, mask, class_weights)
        losses.append(loss.item())
        paths = model.decode(emissions, mask)
        for i, path in enumerate(paths):
            length = int(mask[i].sum())
            preds.extend(path[:length])
            golds.extend(labels[i, :length].tolist())
    metrics = calculate_metrics(golds, preds)
    metrics["loss"] = float(np.mean(losses))
    if hasattr(model, "graph_diagnostics"):
        metrics.update(model.graph_diagnostics())
    report = classification_report(
        golds, preds, labels=list(range(NUM_LABELS)), target_names=LABELS,
        zero_division=0, output_dict=True,
    )
    return metrics, report, golds, preds


def parameter_groups(model):
    backbone = []
    head = []
    for name, parameter in model.named_parameters():
        if name.startswith("backbone."):
            backbone.append(parameter)
        else:
            head.append(parameter)
    return [
        {"params": backbone, "lr": CFG.backbone_lr, "weight_decay": CFG.weight_decay},
        {"params": head, "lr": CFG.head_lr, "weight_decay": CFG.weight_decay},
    ]

def train_model(model, train_loader, dev_loader, run_name, seed):
    model.to(DEVICE)
    optimizer = AdamW(parameter_groups(model))
    steps_per_epoch = math.ceil(len(train_loader) / CFG.grad_accum_steps)
    total_steps = steps_per_epoch * MAX_EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer, int(total_steps * CFG.warmup_ratio), total_steps
    )
    scaler = torch.amp.GradScaler("cuda", enabled=CFG.use_amp)
    checkpoint_path = ARTIFACT_DIR / f"{run_name}_seed{seed}.pt"
    history, best_f1, stale_epochs = [], -1.0, 0

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        backbone_trainable = epoch > CFG.freeze_backbone_epochs
        model.set_backbone_trainable(backbone_trainable)
        optimizer.zero_grad(set_to_none=True)
        epoch_losses = []
        progress = tqdm(train_loader, desc=f"{run_name} seed={seed} epoch={epoch}")
        for step, batch in enumerate(progress, start=1):
            labels = batch["word_labels"].to(DEVICE)
            mask = batch["word_mask"].to(DEVICE)
            with torch.amp.autocast(device_type="cuda", enabled=CFG.use_amp):
                emissions = model(batch)
                loss = model.loss(emissions, labels, mask, class_weights)
                scaled_loss = loss / CFG.grad_accum_steps
            scaler.scale(scaled_loss).backward()
            if step % CFG.grad_accum_steps == 0 or step == len(train_loader):
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scale_before = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                if scaler.get_scale() >= scale_before:
                    scheduler.step()
                optimizer.zero_grad(set_to_none=True)
            epoch_losses.append(loss.item())
            progress.set_postfix(loss=f"{np.mean(epoch_losses[-20:]):.4f}")

        dev_metrics, _, _, _ = evaluate(model, dev_loader)
        row = {
            "epoch": epoch,
            "backbone_trainable": backbone_trainable,
            "train_loss": float(np.mean(epoch_losses)),
            **{f"dev_{k}": v for k, v in dev_metrics.items()},
        }
        history.append(row)
        print(row)

        current_f1 = dev_metrics["entity_macro_f1"]
        if current_f1 > best_f1 + CFG.min_delta:
            best_f1, stale_epochs = current_f1, 0
            torch.save(model.state_dict(), checkpoint_path)
        else:
            stale_epochs += 1
            if stale_epochs >= PATIENCE:
                print(f"Early stopping at epoch {epoch}.")
                break

    state = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    model.load_state_dict(state, strict=True)
    model.to(DEVICE).eval()
    pd.DataFrame(history).to_csv(ARTIFACT_DIR / f"{run_name}_seed{seed}_history.csv", index=False)
    return model, checkpoint_path, history


,label,train_words,weight
0,O,31,1.000000
1,S-MENU_NM,5994,0.270638
2,S-MENU_CNT,2126,0.306496
3,S-MENU_NUM,94,3.000000
4,S-MENU_UNITPRICE,629,0.578035
5,S-MENU_PRICE,2236,0.302234
6,S-MENU_DISCOUNT_PRICE,355,0.903049
7,S-SUBTOTAL,1187,0.388415
8,S-DISCOUNT,164,1.783996
9,S-TAX,1022,0.421619


## 7. Multi-Seed Training and Final Evaluation

This section performs five independent training runs with seeds `13`, `42`, `2026`, `7`, and `123`. Each run starts from pretrained LayoutLMv3 weights with a newly initialized classifier and CRF.

After development-based checkpoint selection, the selected model is evaluated once on the development and test splits. The notebook records per-seed metrics, training duration, peak GPU memory, parameter counts, checkpoint paths, and classification reports. Aggregate results are reported as mean ± sample standard deviation across the five seeds.


In [13]:
experiment_rows = []
reports = {}

for seed in SEEDS:
    print("\n" + "=" * 90)
    print(MODEL_NAME, "seed", seed)
    print("=" * 90)

    set_seed(seed)
    train_loader, dev_loader, test_loader = make_loaders(seed)
    torch.cuda.reset_peak_memory_stats()
    model = LayoutLMv3WordCRF()
    total_parameters = sum(parameter.numel() for parameter in model.parameters())
    trainable_parameters = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

    started = time.time()
    model, checkpoint_path, history = train_model(
        model, train_loader, dev_loader, MODEL_NAME, seed,
    )

    dev_metrics, dev_report, _, _ = evaluate(model, dev_loader)
    test_metrics, test_report, _, _ = evaluate(model, test_loader)
    elapsed = time.time() - started
    peak_vram_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)
    best_history_row = max(history, key=lambda row: row["dev_entity_macro_f1"])

    row = {
        "model": MODEL_NAME, "seed": seed, "checkpoint": str(checkpoint_path),
        "backbone_lr": CFG.backbone_lr, "head_lr": CFG.head_lr,
        "epochs_ran": len(history), "best_epoch": int(best_history_row["epoch"]),
        "total_parameters": total_parameters, "trainable_parameters": trainable_parameters,
        "wall_seconds": elapsed, "peak_vram_gb": peak_vram_gb,
        **{f"dev_{key}": value for key, value in dev_metrics.items()},
        **{f"test_{key}": value for key, value in test_metrics.items()},
    }
    experiment_rows.append(row)
    reports[f"{MODEL_NAME}_seed{seed}"] = {"dev": dev_report, "test": test_report}
    print(pd.Series(row))

    del model, train_loader, dev_loader, test_loader
    gc.collect()
    torch.cuda.empty_cache()

results_df = pd.DataFrame(experiment_rows).sort_values("seed").reset_index(drop=True)
results_df.to_csv(ARTIFACT_DIR / "experiment_results.csv", index=False)
(ARTIFACT_DIR / "classification_reports.json").write_text(
    json.dumps(reports, ensure_ascii=False, indent=2), encoding="utf-8"
)


layoutlmv3_word_crf seed 13


model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

LayoutLMv3Model LOAD REPORT from: microsoft/layoutlmv3-base
Key                                | Status     |  | 
-----------------------------------+------------+--+-
layoutlmv3.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


layoutlmv3_word_crf seed=13 epoch=1:   0%|          | 0/400 [00:00<?, ?it/s]

...

{'epoch': 14, 'backbone_trainable': True, 'train_loss': 0.01541532627750712, 'dev_entity_macro_precision': 0.944711239790521, 'dev_entity_macro_recall': 0.9245788858927975, 'dev_entity_macro_f1': 0.9254896845954755, 'dev_entity_micro_f1': 0.9832913710231174, 'dev_weighted_f1': 0.9811126601308476, 'dev_loss': 0.08819810736371438}


evaluate:   0%|          | 0/50 [00:00<?, ?it/s]

evaluate:   0%|          | 0/50 [00:00<?, ?it/s]

model                                                        layoutlmv3_word_crf
seed                                                                         123
checkpoint                     /kaggle/working/receipt_kie_baseline_artifacts...
backbone_lr                                                              0.00001
head_lr                                                                   0.0001
epochs_ran                                                                    14
best_epoch                                                                    13
total_parameters                                                       125341986
trainable_parameters                                                   125341986
wall_seconds                                                         1732.856951
peak_vram_gb                                                            4.281394
dev_entity_macro_precision                                              0.946015
dev_entity_macro_recall     

34842

In [14]:
def mean_std_text(values, digits=4):
    values = pd.Series(values, dtype=float).dropna()
    mean = values.mean()
    if len(values) < 2:
        return f"{mean:.{digits}f} (1 seed)"
    return f"{mean:.{digits}f} ± {values.std(ddof=1):.{digits}f}"


group = results_df[results_df["model"] == MODEL_NAME]
final_table = pd.DataFrame([{
    "Model": MODEL_NAME,
    "Seeds": len(group),
    "Dev Entity Macro F1": mean_std_text(group["dev_entity_macro_f1"]),
    "Test Entity Macro F1": mean_std_text(group["test_entity_macro_f1"]),
    "Test Entity Micro F1": mean_std_text(group["test_entity_micro_f1"]),
    "Test Weighted F1": mean_std_text(group["test_weighted_f1"]),
    "Best epoch": mean_std_text(group["best_epoch"], digits=1),
    "Wall time (min)": mean_std_text(group["wall_seconds"] / 60, digits=1),
    "Peak VRAM (GiB)": mean_std_text(group["peak_vram_gb"], digits=2),
    "Parameters (M)": f"{group['total_parameters'].iloc[0] / 1e6:.2f}",
}])
final_table.to_csv(ARTIFACT_DIR / "final_results_table.csv", index=False)
print("FINAL TABLE — baseline")
display(final_table)

numeric_summary = results_df.groupby("model").agg(
    seeds=("seed", "count"),
    dev_macro_f1_mean=("dev_entity_macro_f1", "mean"),
    dev_macro_f1_std=("dev_entity_macro_f1", "std"),
    test_macro_f1_mean=("test_entity_macro_f1", "mean"),
    test_macro_f1_std=("test_entity_macro_f1", "std"),
    test_micro_f1_mean=("test_entity_micro_f1", "mean"),
    test_micro_f1_std=("test_entity_micro_f1", "std"),
    test_weighted_f1_mean=("test_weighted_f1", "mean"),
    wall_minutes_mean=("wall_seconds", lambda values: values.mean() / 60),
    peak_vram_gb_mean=("peak_vram_gb", "mean"),
).reset_index()
numeric_summary.to_csv(ARTIFACT_DIR / "numeric_summary.csv", index=False)

deploy_row = group.loc[group["dev_entity_macro_f1"].idxmax()]
deploy_checkpoint = ARTIFACT_DIR / "layoutlmv3_word_crf_best.pt"
shutil.copy2(deploy_row["checkpoint"], deploy_checkpoint)
(ARTIFACT_DIR / "selected_deploy_run.json").write_text(
    json.dumps(deploy_row.to_dict(), ensure_ascii=False, indent=2, default=str), encoding="utf-8"
)
print("Deploy checkpoint selected by dev F1:", deploy_checkpoint)
display(deploy_row[["seed", "best_epoch", "dev_entity_macro_f1", "test_entity_macro_f1"]])

if not CFG.keep_all_checkpoints:
    for checkpoint in ARTIFACT_DIR.glob("*_seed*.pt"):
        checkpoint.unlink()
    print("Removed per-seed checkpoints; retained the dev-selected deploy checkpoint.")

FINAL TABLE — baseline


,Model,Seeds,Dev Entity Macro F1,Test Entity Macro F1,Test Entity Micro F1,Test Weighted F1,Best epoch,Wall time (min),Peak VRAM (GiB),Parameters (M)
0,layoutlmv3_word_crf,5,0.9268 ± 0.0073,0.9276 ± 0.0176,0.9717 ± 0.0033,0.9691 ± 0.0033,10.6 ± 1.5,28.3 ± 1.0,4.28 ± 0.01,125.34


Deploy checkpoint selected by dev F1: /kaggle/working/receipt_kie_baseline_artifacts/layoutlmv3_word_crf_best.pt


seed                          42
best_epoch                    10
dev_entity_macro_f1     0.934076
test_entity_macro_f1     0.94048
Name: 2, dtype: object

## 8. Per-Class Performance Analysis

This section aggregates precision, recall, F1, and support for every entity class across all seeds. Classes are ordered by mean F1 to expose rare or difficult receipt fields that may be hidden by aggregate metrics.


In [15]:
per_class_rows = []
for run_key, split_reports in reports.items():
    model_name, seed_text = run_key.rsplit("_seed", 1)
    test_report = split_reports["test"]
    for label in LABELS[1:]:
        values = test_report.get(label, {})
        per_class_rows.append({
            "model": model_name,
            "seed": int(seed_text),
            "label": label,
            "precision": values.get("precision", 0.0),
            "recall": values.get("recall", 0.0),
            "f1": values.get("f1-score", 0.0),
            "support": values.get("support", 0.0),
        })

per_class_df = pd.DataFrame(per_class_rows)
per_class_df.to_csv(ARTIFACT_DIR / "per_class_results_by_seed.csv", index=False)
per_class_summary = per_class_df.groupby(["model", "label"]).agg(
    f1_mean=("f1", "mean"), f1_std=("f1", "std"),
    precision_mean=("precision", "mean"), recall_mean=("recall", "mean"),
    support=("support", "first"), seeds=("seed", "count"),
).reset_index()
per_class_summary.to_csv(ARTIFACT_DIR / "per_class_summary.csv", index=False)

model_class_summary = per_class_summary[
    per_class_summary["model"] == MODEL_NAME
].sort_values("f1_mean")
print("BASELINE CLASSES — weakest first")
display(model_class_summary)

BASELINE CLASSES — weakest first


,model,label,f1_mean,f1_std,precision_mean,recall_mean,support,seeds
4,layoutlmv3_word_crf,S-EMONEY_PAYMENT,0.580000,0.175752,0.478333,0.750000,4.0,5
13,layoutlmv3_word_crf,S-OTHER,0.796223,0.047616,0.874472,0.731707,41.0,5
6,layoutlmv3_word_crf,S-MENUTYPE_CNT,0.796746,0.023477,0.948214,0.694118,17.0,5
0,layoutlmv3_word_crf,S-CARD_PAYMENT,0.940142,0.039471,0.947004,0.937255,51.0,5
10,layoutlmv3_word_crf,S-MENU_NUM,0.948421,0.073534,1.000000,0.909091,11.0,5
8,layoutlmv3_word_crf,S-MENU_DISCOUNT_PRICE,0.949840,0.042826,0.917123,0.986667,30.0,5
5,layoutlmv3_word_crf,S-MENUQTY_CNT,0.952363,0.006541,0.922096,0.985075,67.0,5
1,layoutlmv3_word_crf,S-CASH,0.960642,0.009109,0.981746,0.940541,148.0,5
3,layoutlmv3_word_crf,S-DISCOUNT,0.961263,0.027061,0.987500,0.937500,16.0,5
12,layoutlmv3_word_crf,S-MENU_UNITPRICE,0.971299,0.014905,0.966218,0.976812,69.0,5


## 9. Reproducibility Metadata and Artifact Packaging

This section packages the complete baseline experiment for reproducible evaluation and deployment. The artifact contains:

- The checkpoint selected for deployment by development-set Entity Macro F1.
- The selected checkpoint from each training seed.
- The LayoutLMv3 processor and label mappings.
- Per-seed results, aggregate tables, classification reports, and training histories.
- Model configuration, dataset split sizes, library versions, and selection protocol.
- SHA-256 checksums for checkpoint integrity verification.

The resulting archive is written to `/kaggle/working/receipt_kie_baseline_artifacts.zip`.


In [16]:
metadata = {
    "architecture": "LayoutLMv3WordCRF",
    "selection_protocol": "Best checkpoint per seed selected by dev entity macro F1; test evaluated once afterward",
    "reporting_protocol": "Baseline rerun with fixed seeds; mean and sample standard deviation",
    "metric_definition": "Macro F1 over 18 entity labels, excluding O, oracle CORD text/bboxes",
    "config": asdict(CFG),
    "profile": CFG.profile,
    "seeds": SEEDS,
    "label2id": LABEL2ID,
    "id2label": ID2LABEL,
    "dataset_root_name": CORD_ROOT.name,
    "split_documents": {"train": len(train_records), "dev": len(dev_records), "test": len(test_records)},
    "versions": {
        "python": platform.python_version(), "torch": torch.__version__,
        "cuda": torch.version.cuda, "transformers": transformers.__version__,
        "sklearn": sklearn.__version__,
    },
}
(ARTIFACT_DIR / "model_metadata.json").write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8"
)
(ARTIFACT_DIR / "label_map.json").write_text(
    json.dumps({"label2id": LABEL2ID, "id2label": ID2LABEL}, ensure_ascii=False, indent=2), encoding="utf-8"
)
processor.save_pretrained(ARTIFACT_DIR / "processor")

checkpoint_name = "layoutlmv3_word_crf_best.pt"
checkpoint_path = ARTIFACT_DIR / checkpoint_name
assert checkpoint_path.is_file(), "Không có deploy checkpoint; không được tạo ZIP thiếu model."
seed_checkpoint_names = [f"{MODEL_NAME}_seed{seed}.pt" for seed in SEEDS]
checkpoint_names = [checkpoint_name, *seed_checkpoint_names]
checkpoint_hashes = {}
for current_name in checkpoint_names:
    current_path = ARTIFACT_DIR / current_name
    assert current_path.is_file(), f"Thiếu checkpoint cần đóng gói: {current_name}"
    digest = hashlib.sha256()
    with current_path.open("rb") as checkpoint_file:
        for chunk in iter(lambda: checkpoint_file.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    checkpoint_hashes[current_name] = digest.hexdigest()
checkpoint_hash = checkpoint_hashes[checkpoint_name]
(ARTIFACT_DIR / "SHA256SUMS.txt").write_text(
    "".join(f"{digest}  {filename}\n" for filename, digest in checkpoint_hashes.items()),
    encoding="utf-8",
)

required_artifacts = [
    checkpoint_name, "selected_deploy_run.json", "model_metadata.json",
    "label_map.json", "final_results_table.csv", "experiment_results.csv",
    "classification_reports.json", "SHA256SUMS.txt",
]
required_artifacts.extend(seed_checkpoint_names)
missing_artifacts = [name for name in required_artifacts if not (ARTIFACT_DIR / name).exists()]
assert not missing_artifacts, f"Artifact còn thiếu: {missing_artifacts}"
(ARTIFACT_DIR / "ARTIFACT_README.txt").write_text(
    "CORD oracle-OCR LayoutLMv3 Word-CRF artifact.\n"
    "Checkpoint was selected only by dev Entity Macro F1.\n"
    "Load model weights with strict=True using the baseline architecture in Result_Baseline.ipynb.\n"
    f"Checkpoint SHA256: {checkpoint_hash}\n",
    encoding="utf-8",
)

archive = shutil.make_archive("/kaggle/working/receipt_kie_baseline_artifacts", "zip", ARTIFACT_DIR)
assert Path(archive).is_file() and Path(archive).stat().st_size > 0
print("Deploy checkpoint:", checkpoint_path)
print("Artifact archive:", archive)
print("Tải file ZIP từ tab Output của Kaggle sau khi notebook chạy xong.")

Deploy checkpoint: /kaggle/working/receipt_kie_baseline_artifacts/layoutlmv3_word_crf_best.pt
Artifact archive: /kaggle/working/receipt_kie_baseline_artifacts.zip
Tải file ZIP từ tab Output của Kaggle sau khi notebook chạy xong.


## 10. Reporting Scope

Results from the `paper` profile should be reported as:

```text
On the CORD test split with oracle text and bounding boxes, LayoutLMv3 with a word-level CRF achieved an Entity Macro F1 of mean ± sample standard deviation across five independent seeds.
```

The reported Entity Macro F1 covers 18 semantic entity labels and excludes `O`. Because the experiment uses reference CORD text and bounding boxes, it evaluates the KIE component rather than an end-to-end OCR-to-extraction system. A complete end-to-end evaluation would additionally require OCR character or word error rate, entity F1 from OCR-generated inputs, receipt-level exact match, and inference latency.

The downloadable archive is `/kaggle/working/receipt_kie_baseline_artifacts.zip`.
